In [ ]:
import os
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from itertools import combinations
from sklearn.preprocessing import MinMaxScaler
import logging
import sys

# --- Reusable Functions ---
from ReusableFunctions.DataPreprocessing import DataPreprocessing
from ReusableFunctions.EvaluationMetrics import EvaluationMetrics as EM
from reproducibility_settings import set_global_seed
from ReusableFunctions.RecordBestModel import record_best_models

# -------------------------------
# Reproducibility
# -------------------------------
set_global_seed(seed=42, framework='torch')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

# -------------------------------
# LSTM Model
# -------------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout, activation):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        activation_map = {"relu": nn.ReLU, "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}
        self.act = activation_map[activation]()
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.act(out[:, -1, :])  # last timestep
        return self.fc(out)

# -------------------------------
# Dataset Preparation
# -------------------------------
def prepare_data(df, selected_features, window_size, forecast_window, ticker):
    print(f"⚙️ Generating dataset for {ticker} (w={window_size}, f={forecast_window})...")
    processor = DataPreprocessing(ticker=ticker)

    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df[selected_features])

    X, y = processor.create_windowed_data(scaled_data, window_size, forecast_window)
    X_train, X_val, _, y_train, y_val, _ = processor.split_dataset(X, y)

    scaler_y = MinMaxScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
    y_val_scaled = scaler_y.transform(y_val.reshape(-1, 1))

    return {
        "X_train": X_train,
        "X_val": X_val,
        "y_train": y_train_scaled,
        "y_val": y_val_scaled,
        "scaler_y": scaler_y
    }

# -------------------------------
# Training + Evaluation
# -------------------------------
def train_and_evaluate_model(model, train_loader, val_data, optimizer, criterion, scaler_y,
                             forecast_window, epochs, patience=0.2):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    actual_patience = int(patience * epochs) if isinstance(patience, float) else patience

    X_val_tensor = torch.tensor(val_data[0], dtype=torch.float32).to(device)
    y_val_tensor = torch.tensor(val_data[1], dtype=torch.float32).to(device)

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb).squeeze(), yb.squeeze())
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds_scaled = model(X_val_tensor).squeeze()
            val_loss = criterion(val_preds_scaled, y_val_tensor.squeeze()).item()

        if val_loss < best_val_loss:
            best_val_loss, epochs_no_improve = val_loss, 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= actual_patience:
                print(f"Early stopping triggered at epoch {epoch+1}.")
                break

    model.eval()
    with torch.no_grad():
        val_preds_scaled = model(X_val_tensor).cpu().numpy()
        val_preds_unscaled = scaler_y.inverse_transform(val_preds_scaled.reshape(-1, 1))
        y_val_true_unscaled = scaler_y.inverse_transform(val_data[1])

    r2 = EM.r2(y_val_true_unscaled, val_preds_unscaled)
    rmse, mape, acc = (np.nan, np.nan, np.nan) if np.isnan(r2) or np.isinf(r2) else (
        EM.rmse(y_val_true_unscaled, val_preds_unscaled),
        EM.mape(y_val_true_unscaled, val_preds_unscaled),
        EM.accuracy(y_val_true_unscaled, val_preds_unscaled),
    )
    profit_index = EM.profitability_index(y_val_true_unscaled, val_preds_unscaled, forecast_window)

    del X_val_tensor, y_val_tensor
    torch.cuda.empty_cache()
    return rmse, mape, r2, acc, profit_index

# -------------------------------
# Main Execution (Fixed Params)
# -------------------------------
if __name__ == "__main__":
    ticker = 'QCOM'
    os.makedirs('stock_results', exist_ok=True)
    if not os.path.exists(f'stock_results/{ticker}_LSTM_results.csv'):
        with open(f'stock_results/{ticker}_LSTM_results.csv', 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                "Indicators", "Hidden Size", "Dropout", "LR", "Batch Size",
                "Window Size", "Forecast Window", "Epochs", "Activation",
                "RMSE", "MAPE", "R2", "Accuracy", "Profit Index"
            ])

    data_processor = DataPreprocessing(ticker)
    df_with_all_indicators = data_processor.add_technical_indicators()

    selected_base_indicators = ['20MA', '50MA', 'RSI', 'MACD', 'Upper_BB',
                                'Lower_BB', 'CCI', 'ATR', 'Williams_%R', 'OBV']
    all_combinations = list(combinations(selected_base_indicators, 6))
    window_forecast_combos = [(5, 1)]

    # 🔹 Fixed hyperparameters
    fixed_hparams = {
        "hidden_size": 128,
        "dropout": 0.2,
        "lr": 0.0005,
        "batch_size": 64,
        "activation": "tanh",
        "epochs": 100,
        "num_layers": 1
    }

    for i, indicator_combo in enumerate(all_combinations):
        for j, (window_size, forecast_window) in enumerate(window_forecast_combos):
            print(f"\n=== [{i+1}/{len(all_combinations)}] Combo: {indicator_combo}, (w={window_size}, f={forecast_window}) ===")
            
            data = prepare_data(
                df_with_all_indicators,
                ['Close'] + list(indicator_combo),
                window_size,
                forecast_window,
                ticker
            )

            train_loader = DataLoader(
                TensorDataset(torch.tensor(data["X_train"], dtype=torch.float32),
                              torch.tensor(data["y_train"], dtype=torch.float32)),
                batch_size=fixed_hparams["batch_size"],
                shuffle=False
            )

            model = LSTMModel(
                input_size=data["X_train"].shape[2],
                hidden_size=fixed_hparams["hidden_size"],
                num_layers=fixed_hparams["num_layers"],
                dropout=fixed_hparams["dropout"],
                activation=fixed_hparams["activation"]
            ).to(device)

            optimizer = optim.Adam(model.parameters(), lr=fixed_hparams["lr"])
            criterion = nn.MSELoss()

            rmse, mape, r2, acc, profit_index = train_and_evaluate_model(
                model, train_loader, (data["X_val"], data["y_val"]),
                optimizer, criterion, data["scaler_y"], forecast_window, fixed_hparams["epochs"]
            )

            with open(f'stock_results/{ticker}_LSTM_results.csv', 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([
                    ', '.join(indicator_combo),
                    fixed_hparams["hidden_size"], fixed_hparams["dropout"],
                    fixed_hparams["lr"], fixed_hparams["batch_size"],
                    window_size, forecast_window, fixed_hparams["epochs"],
                    fixed_hparams["activation"],
                    rmse, mape, r2, acc, profit_index
                ])

    record_best_models(f'stock_results/{ticker}_LSTM_results.csv')
